## Notebook Purpose

This notebook explores Google's [Places Api](https://developers.google.com/maps/documentation/places/web-service/overview?_gl=1*1h2my3u*_up*MQ..*_ga*MTQ1OTU0Mjc1NC4xNzY1MjMyMzIx*_ga_NRWSTWS78N*czE3NjUyMzIzMjEkbzEkZzEkdDE3NjUyMzIzNzkkajIkbDAkaDA.*_ga_SM8HXJ53K2*czE3NjUyMzIzMjEkbzEkZzAkdDE3NjUyMzIzMjEkajYwJGwwJGgw)

In [10]:
import requests
import os
import json
import pandas as pd

## Test Places Search requests for cafes in London

The [documentation](https://developers.google.com/maps/documentation/places/web-service/text-search?_gl=1*1is1mlh*_up*MQ..*_ga*MTE1NjAyMDMwNi4xNzY1Mjk4MzM2*_ga_NRWSTWS78N*czE3NjUyOTgzMzYkbzEkZzAkdDE3NjUyOTgzMzckajU5JGwwJGgw*_ga_SM8HXJ53K2*czE3NjUyOTgzMzYkbzEkZzAkdDE3NjUyOTgzMzYkajYwJGwwJGgw) was followed in order to create this test.

In [3]:
# Constants
places_url = 'https://places.googleapis.com/v1/places:searchText'
API_KEY = os.getenv('API_KEY')

In [4]:
headers = {
    'Content-Type': 'application/json',
    'X-Goog-API-Key': API_KEY,
    'X-Goog-FieldMask': '*',
    # IMPORTANT: using '*' for FieldMask returns all fields. I am only using this for the test - don't use when we know what fields we want returned because it could increase costs.
}

data = {
    'textQuery': 'cafes in Holborn, London, England'
}

response = requests.post(url=places_url, headers=headers, data=json.dumps(data))

if response.status_code == 200:
    print('Success!')
else:
    print(f'Error: {response.status_code}: {response.text}')

Error: 403: {
  "error": {
    "code": 403,
    "message": "Method doesn't allow unregistered callers (callers without established identity). Please use API Key or other form of API consumer identity to call this API.",
    "status": "PERMISSION_DENIED"
  }
}



In [5]:
response.json()

{'error': {'code': 403,
  'message': "Method doesn't allow unregistered callers (callers without established identity). Please use API Key or other form of API consumer identity to call this API.",
  'status': 'PERMISSION_DENIED'}}

## Test Places Aggregate API response

[This guide](https://developers.google.com/maps/documentation/places-aggregate/make-your-first-request?_gl=1*1kb9yk0*_up*MQ..*_ga*ODM0NTQyNDE5LjE3NjUyODM1NzE.*_ga_SM8HXJ53K2*czE3NjUyODM1NzEkbzEkZzAkdDE3NjUyODM1NzEkajYwJGwwJGgw*_ga_NRWSTWS78N*czE3NjUyODM1NzEkbzEkZzAkdDE3NjUyODM1NzEkajYwJGwwJGgw) shows how to set up a request.

This example aims to count how many cafes there are in the LSE postcode 'region'. 

In [6]:
# Constants
url = 'https://areainsights.googleapis.com/v1:computeInsights'
API_KEY = os.getenv('API_KEY')
place_id = 'places/ChIJWx2zXLUEdkgRNoC9_WcA2NE'
# Place ID for the LSE postcode (WC2A2AE), found using place ID Finder: https://developers.google.com/maps/documentation/javascript/examples/places-placeid-finder?_gl=1*1wj64qj*_up*MQ..*_ga*MTAwMjkzMjcwNi4xNzY1Mjg0NDM3*_ga_NRWSTWS78N*czE3NjUyODQ0MzckbzEkZzEkdDE3NjUyODYxNzAkajYwJGwwJGgw*_ga_SM8HXJ53K2*czE3NjUyODQ0MzckbzEkZzAkdDE3NjUyODYxMzYkajYwJGwwJGgw

In [7]:
headers = {
    'X-Goog-Api-Key': API_KEY,
    'Content-Type': 'application/json',
}

payload = {
    'insights': [
        'INSIGHT_COUNT'
    ],
    'filter':{
        'locationFilter': {
            'region': {
                'place': place_id
            }
        },
        'typeFilter':{
            'includedTypes':[
                'cafe'
            ]
        },
        'operatingStatus':[
            'OPERATING_STATUS_OPERATIONAL'
        ]
    }      
}

response = requests.post(url, headers=headers, json=payload)
print(response.status_code)

403


### Notes on API request

The code above adapts the guide provided by google (which is written using curl) to python. Where curl uses `--data`, I use `payload`.

In [8]:
# Inspect 403 response status code
try:
    error_data = response.json()
    print(json.dumps(error_data, indent=2))
except ValueError:
    print("Response is not JSON:")
    print(response.text)

{
  "error": {
    "code": 403,
    "message": "Method doesn't allow unregistered callers (callers without established identity). Please use API Key or other form of API consumer identity to call this API.",
    "status": "PERMISSION_DENIED"
  }
}


### Testing Overpass API

In [13]:
url = "https://overpass-api.de/api/interpreter"

query = """
[out:json][timeout:25];
(
  node["man_made"="surveillance"](51.50,-0.15,51.54,-0.02);
  way["man_made"="surveillance"](51.50,-0.15,51.54,-0.02);
  relation["man_made"="surveillance"](51.50,-0.15,51.54,-0.02);
);
out center;
"""

# Overpass expects form data, not JSON
payload = {"data": query}

response = requests.post(url, data=payload)
print("Status code:", response.status_code)

data = response.json()
elements = data["elements"]
print("Number of surveillance objects:", len(elements))

Status code: 200
Number of surveillance objects: 648


- the coordinates I used here were for a crid of London: 
    South → near Vauxhall / Lambeth
    North → near Islington / King’s Cross
    West → near Westminster
    East → near Shoreditch
- ⚠️ to be careful with Overpass API: It is free and open (no API key needed) so rate limits exist. Meaning we can probably make 1–3 queries total (small areas of London). Large bounding boxes (e.g., all of London) may time out, but borough-sized should be fine.
- **Explanation:** we filter for objects tagged man_made="surveillance", which is the standard OpenStreetMap tag for public-facing CCTV cameras. The query also specifies a geographic bounding box corresponding to our chosen area of London, ensuring we only retrieve cameras within that region. We include nodes, ways, and relations in the search because surveillance objects can be mapped in multiple forms. The `[out:json]` directive tells Overpass to return results in JSON format, which we can easily parse in Python. Finally, the `out center;` command ensures that ways and relations, which do not have a single point geometry (a lon/lat), are assigned a representative centre point so their locations can be plotted and used consistently in our analysis.

In [14]:
# Convert to a simple DataFrame: id, type, lat, lon, tags
rows = []
for el in elements:
    # nodes have lat/lon directly; ways/relations have a 'center'
    if el["type"] == "node":
        lat = el.get("lat")
        lon = el.get("lon")
    else:
        center = el.get("center", {})
        lat = center.get("lat")
        lon = center.get("lon")

    rows.append({
        "id": el["id"],
        "osm_type": el["type"],
        "lat": lat,
        "lon": lon,
        "tags": el.get("tags", {})
    })

cctv_df = pd.DataFrame(rows)
cctv_df.head()

,id,osm_type,lat,lon,tags
0,94247210,node,51.519739,-0.056101,{'man_made': 'surveillance'}
1,96219132,node,51.520538,-0.052134,{'man_made': 'surveillance'}
2,264522811,node,51.504974,-0.083752,"{'camera:mount': 'ceiling', 'camera:type': 'fi..."
3,296631202,node,51.505896,-0.082766,"{'camera:mount': 'ceiling', 'camera:type': 'do..."
4,305955790,node,51.502742,-0.110779,"{'created_by': 'Potlatch 0.10e', 'man_made': '..."


- Here, we are looping through each element from Overpass, extracting either: `lat/lon` directly (if it’s a node), or
`center.lat/center.lon` (if it’s a way or relation). Then we append all the cleaned records to a list, and converts that list into a DataFrame. 